# IPEDS → SOC Method Validation

## Purpose
Validates the same CIP→SOC method used for the manuscript postsecondary alignment analysis.

This notebook is intentionally separate from the publication notebook so diagnostic work does not clutter the manuscript workflow.

## Current source of truth
`Master_CIP_SOC_Matched_Output_3.xlsx`

The master file now defines:
- Healthcare
- Manufacturing
- Construction
- Semiconductor occupations through `Semiconductor Flag = Yes`
- 5-star status through `Sum of Occupation Rating by Ed Lvl1 = 5`
- CIP→SOC relationships

The validation universe uses the same sector order as the manuscript:
Healthcare and Social Assistance → Manufacturing → Construction → Semiconductors.


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ============================================================
# COMPUTER — CHANGE ONLY THIS VALUE
# ============================================================
# "W" = work computer
# "P" = personal computer
# "S" = Shared Drive source
COMPUTER = "W"

MASTER_PATHS = {
    "W": Path(
        r"C:\Users\301533\Documents\Walmart\IPEDS\Files"
        r"\Master_CIP_SOC_Matched_Output_3.xlsx"
    ),
    "S": Path(
        r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
        r"\SOC NAICS List\Economic Analysis\Walmart_LMI_Analysis\Current\Python"
        r"\Master_CIP_SOC_Matched_Output_3.xlsx"
    ),
}

BASE_FOLDERS = {
    "W": Path(r"C:\Users\301533\Documents\Walmart\IPEDS\Files"),
    "S": Path(
        r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
        r"\SOC NAICS List\Economic Analysis\Walmart_LMI_Analysis\Current\Python"
    ),
}

OUTPUT_FOLDER = Path(
    r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)

key = str(COMPUTER).strip().upper()
if key not in MASTER_PATHS:
    raise ValueError('COMPUTER must be "W", "P", or "S".')

MASTER_FILE = MASTER_PATHS[key]
BASE_FOLDER = BASE_FOLDERS[key]

SECTOR_ORDER = [
    "Healthcare and Social Assistance",
    "Manufacturing",
    "Construction",
    "Semiconductors",
]

FIVE_STAR = 5

def clean_soc(value):
    if pd.isna(value):
        return np.nan
    digits = re.sub(r"\D", "", str(value))
    if len(digits) >= 6:
        return f"{digits[:2]}-{digits[2:6]}"
    return np.nan

def clean_cip(value):
    if pd.isna(value):
        return np.nan
    is_numeric_input = isinstance(value, (int, float, np.integer, np.floating))
    s = str(value).strip().replace('="', '').replace('"', '').replace("'", "")
    s = re.sub(r"[^0-9.]", "", s)
    if not s:
        return np.nan
    if "." in s:
        left, right = s.split(".", 1)
        right = re.sub(r"\D", "", right)
        if is_numeric_input and 1 <= len(right) <= 4:
            return f"{left.zfill(2)}.{right.ljust(4, '0')[:4]}"
        if len(right) >= 4:
            return f"{left.zfill(2)}.{right[:4]}"
    digits = re.sub(r"\D", "", s)
    if len(digits) == 6:
        return f"{digits[:2]}.{digits[2:]}"
    return np.nan

def to_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

def load_master():
    if not MASTER_FILE.exists():
        raise FileNotFoundError(
            f"Master file not found:\n{MASTER_FILE}\n\n"
            "Change COMPUTER or update MASTER_PATHS."
        )

    master = pd.read_excel(
        MASTER_FILE,
        sheet_name="Master_Matched_Data"
    ).copy()

    required = [
        "SOC_Code",
        "Job Title",
        "Employment ('25)",
        "Education Level",
        "Projected Ann. Job Openings ('25 - '27)",
        "Projected Ann. Job Chg. ('25-'27)",
        "Average Wage ('25)",
        "Sum of Occupation Rating by Ed Lvl1",
        "Industry",
        "Semiconductor Flag",
        "CIP_Code",
        "CIP_Title",
    ]

    missing = [c for c in required if c not in master.columns]
    if missing:
        raise KeyError(f"Master workbook is missing required columns: {missing}")

    master["soc"] = master["SOC_Code"].map(clean_soc)
    master["cip_current"] = master["CIP_Code"].map(clean_cip)
    master["stars"] = pd.to_numeric(
        master["Sum of Occupation Rating by Ed Lvl1"],
        errors="coerce"
    )
    master["semiconductor_yes"] = (
        master["Semiconductor Flag"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("yes")
    )

    return master

def sectorize(master, five_star_only=True, require_cip=False):
    """
    Return the occupation/CIP universe in the manuscript sector order.

    Manufacturing includes the full Manufacturing universe.
    Semiconductors are also shown separately as a flagged subset of Manufacturing.
    """

    x = master.copy()

    if five_star_only:
        x = x[x["stars"].eq(FIVE_STAR)].copy()

    x = x[x["soc"].notna()].copy()

    if require_cip:
        x = x[x["cip_current"].notna()].copy()

    pieces = []

    definitions = [
        (
            "Healthcare and Social Assistance",
            x["Industry"].eq("Healthcare")
        ),
        (
            "Manufacturing",
            x["Industry"].eq("Manufacturing")
        ),
        (
            "Construction",
            x["Industry"].eq("Construction")
        ),
        (
            "Semiconductors",
            x["semiconductor_yes"]
        ),
    ]

    for sector, mask in definitions:
        part = x.loc[mask].copy()
        part["sector"] = sector
        pieces.append(part)

    result = pd.concat(pieces, ignore_index=True)

    result["sector"] = pd.Categorical(
        result["sector"],
        categories=SECTOR_ORDER,
        ordered=True
    )

    return result.sort_values(["sector", "soc"]).reset_index(drop=True)

master = load_master()

five_star_universe = sectorize(
    master,
    five_star_only=True,
    require_cip=False
)

counts = (
    five_star_universe
    .drop_duplicates(["sector", "soc"])
    .groupby("sector", observed=False)["soc"]
    .nunique()
    .reindex(SECTOR_ORDER)
)

print("Master:", MASTER_FILE)
print("\nFive-star occupation counts:")
print(counts)

semi_count = int(counts.get("Semiconductors", 0))
if semi_count != 4:
    raise ValueError(
        f"Expected 4 five-star Semiconductor occupations; found {semi_count}. "
        "Review Semiconductor Flag before continuing."
    )


CROSSWALK_FILE = BASE_FOLDER / "Crosswalk.csv"

IPEDS_FILE_MAP = {
    2014: BASE_FOLDER / "CSV_9242026-348.csv",
    2019: BASE_FOLDER / "CSV_9242026-150.csv",
    2025: BASE_FOLDER / "CSV_9202026-120.csv",
}

OUTPUT_FILE = (
    OUTPUT_FOLDER
    / "IPEDS_SOC_Method_Validation_Current_Master.xlsx"
)

for path in [MASTER_FILE, CROSSWALK_FILE, *IPEDS_FILE_MAP.values()]:
    if not path.exists():
        raise FileNotFoundError(path)


Master: C:\Users\301533\Documents\Walmart\IPEDS\Files\Master_CIP_SOC_Matched_Output_3.xlsx

Five-star occupation counts:
sector
Healthcare and Social Assistance    25
Manufacturing                       11
Construction                         9
Semiconductors                       4
Name: soc, dtype: int64


In [2]:
# ============================================================
# MASTER RELATIONSHIP QA
# ============================================================

selected = sectorize(
    master,
    five_star_only=True,
    require_cip=True
).copy()

relationships = (
    selected[
        [
            "sector",
            "soc",
            "Job Title",
            "cip_current",
            "CIP_Title",
        ]
    ]
    .drop_duplicates(
        ["sector", "soc", "cip_current"]
    )
)

cip_breadth = (
    relationships.groupby(
        ["sector", "cip_current"],
        observed=False
    )["soc"]
    .nunique()
    .reset_index(name="selected_socs_per_cip")
)

soc_breadth = (
    relationships.groupby(
        ["sector", "soc", "Job Title"],
        observed=False
    )["cip_current"]
    .nunique()
    .reset_index(name="related_cips")
)

print("5-star SOC counts:")
print(
    relationships
    .drop_duplicates(["sector", "soc"])
    .groupby("sector", observed=False)["soc"]
    .nunique()
    .reindex(SECTOR_ORDER)
)

print("\nCIPs mapping to more than one selected SOC:")
display(
    cip_breadth[
        cip_breadth["selected_socs_per_cip"] > 1
    ].sort_values(
        ["sector", "selected_socs_per_cip"],
        ascending=[True, False]
    )
)


5-star SOC counts:
sector
Healthcare and Social Assistance    25
Manufacturing                       11
Construction                         9
Semiconductors                       4
Name: soc, dtype: int64

CIPs mapping to more than one selected SOC:


,sector,cip_current,selected_socs_per_cip
51,Healthcare and Social Assistance,51.1012,2
76,Manufacturing,14.1001,3
82,Manufacturing,14.4101,2
83,Manufacturing,14.4701,2
164,Construction,46.0201,2
165,Construction,46.0302,2
168,Construction,46.0502,2
220,Semiconductors,14.1001,2


In [3]:
# ============================================================
# CROSSWALK QA
# ============================================================

crosswalk = pd.read_csv(
    CROSSWALK_FILE,
    dtype=str
)

crosswalk["cip2010"] = crosswalk["Original code"].map(clean_cip)
crosswalk["cip_current"] = crosswalk["Current code"].map(clean_cip)

cw6 = crosswalk.dropna(
    subset=["cip2010", "cip_current"]
).copy()

conflicts = (
    cw6.groupby("cip2010")["cip_current"]
    .nunique()
    .reset_index(name="n_current_codes")
)

conflicts = conflicts[
    conflicts["n_current_codes"] > 1
].copy()

print(
    "Detailed CIP codes with more than one current mapping:",
    len(conflicts)
)

display(conflicts.head(25))


Detailed CIP codes with more than one current mapping: 0


,cip2010,n_current_codes


In [4]:
# ============================================================
# EXPORT VALIDATION TABLES
# ============================================================

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    relationships.to_excel(
        writer,
        sheet_name="CIP-SOC Relationships",
        index=False
    )

    cip_breadth.to_excel(
        writer,
        sheet_name="CIP Breadth",
        index=False
    )

    soc_breadth.to_excel(
        writer,
        sheet_name="SOC Breadth",
        index=False
    )

    conflicts.to_excel(
        writer,
        sheet_name="Crosswalk Conflicts",
        index=False
    )

    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False

print("Saved:", OUTPUT_FILE)


Saved: G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis\SOC NAICS List\Excel Sheets\Arielle Tables\IPEDS_SOC_Method_Validation_Current_Master.xlsx
